# Volume 14, Chapter 6: The Bohr–Jessen law (Python companion)

Exercises first, solutions at the end. We test Kronecker–Weyl for the primes, compare the values of $\log|\zeta(0.7+it)|$ with the random Euler product, compute the mean square, watch the characteristic function of the limit law decay, hunt for values of $\zeta$ near prescribed points (Bohr–Courant), and follow the variance as $\sigma\to\frac12$. The computer science application: **evaluating $\zeta$ on a long grid of $t$ at once**, vectorising the Euler–Maclaurin formula with numpy instead of calling a multiprecision routine point by point. Requires `numpy`, `scipy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath
from scipy.stats import ks_2samp

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]
P = primes_upto(10**5).astype(float)
rng = np.random.default_rng(6)

def model_logzeta(sigma, M, primes=P):
    # samples of L(sigma, X) = sum_p -log(1 - X_p p^-sigma)
    L = np.zeros(M, dtype=complex)
    for i in range(0, len(primes), 500):
        pp = primes[i:i + 500]
        X = np.exp(2j * np.pi * rng.random((M, len(pp))))
        L += (-np.log(1 - X * pp ** (-sigma))).sum(axis=1)
    return L

## Exercises

**P6.1 (★) Kronecker–Weyl.** For $t$ uniform on $[T,2T]$ with $T=10^6$ ($10^5$ samples), estimate $\mathbb E\,2^{-itm_1}3^{-itm_2}$ and $\mathbb E\,2^{-itm_1}4^{-itm_2}$ for small $(m_1,m_2)$ (Lemma 6.2 and Exercise 6.3).

**P6.2 (★★) Bohr–Jessen.** Compute $\log|\zeta(0.7+it)|$ at $2000$ random $t\in[10^5,2\cdot10^5]$ with mpmath, compare with $20\,000$ samples of $\log|\zeta(0.7,X)|$ (Kolmogorov–Smirnov), and compare the variances with $v(0.7)=\frac12\sum_p\sum_k\frac1{k^2p^{1.4k}}$ (Theorem 6.5, Figure 6.2).

**P6.3 (★) The mean square.** Estimate $\frac1T\int_T^{2T}|\zeta(0.7+it)|^2dt$ for $T=10^3$ and $T=3\cdot10^3$ with the fast evaluator of P6.5, and compare with $\zeta(1.4)$ (Exercise 6.1).

**P6.4 (★★) The characteristic function.** Compute $\varphi(u)=\prod_{p\le10^5}\mathbb Ee^{iu\Re\ell_p(X_p)}$ for $\sigma=0.7$ and $u\le60$, by numerical integration over the circle, and compare $-\log|\varphi(u)|$ with $u^{1/\sigma}/\log u$ (Theorem 6.8).

**P6.5 (★★) Bohr–Courant, by search.** Write a vectorised evaluator of $\zeta(\sigma+it)$ on a grid of $t$ (Euler–Maclaurin with two correction terms), check it against mpmath, and search $t\in[0,2000]$ for the closest approach of $\zeta(0.7+it)$ to $z=3$, $z=-1+2i$ and $z=0.1$. Compare the proportion of time spent within $0.2$ of $3$ with the model probability (Corollary 6.7).

**P6.6 (★★) Toward the critical line.** Compute $v(\sigma)$ for $\sigma=0.6,0.55,0.52,0.51,0.505$ and compare with $\frac12\log\frac1{\sigma-1/2}$ (Exercise 6.4).

## Solutions

### P6.1

In [ ]:
T = 1e6; t = rng.uniform(T, 2 * T, 100000)
for (a, b) in [(2, 3), (2, 4)]:
    for m in [(1, 0), (1, 1), (2, -1), (1, -1)]:
        v = np.mean(np.exp(-1j * t * (m[0] * math.log(a) + m[1] * math.log(b))))
        print(f"({a},{b}) m={m}: |E| = {abs(v):.3f}")

For $(2,3)$ every non-zero $m$ gives a mean of order $10^{-2.5}$, the Monte Carlo error. For $(2,4)$ the monomial $2^{-2it}4^{it}$ is identically $1$: the pair lives on the curve $z_2=z_1^2$.

### P6.2

In [ ]:
mpmath.mp.dps = 15
ts = rng.uniform(1e5, 2e5, 2000)
t0 = time.time()
data = np.array([math.log(abs(complex(mpmath.zeta(mpmath.mpc(0.7, x))))) for x in ts])
print("mpmath time:", round(time.time() - t0, 1), "s")
model = model_logzeta(0.7, 20000).real
v07 = 0.5 * sum(sum(p ** (-1.4 * k) / k**2 for k in range(1, 30)) for p in P)
print("variance: data", data.var(), " model", model.var(), " v(0.7) (p<=1e5)", v07)
print(ks_2samp(data, model))

The Kolmogorov–Smirnov test does not reject at the $5\%$ level (with $2000$ points the $p$-value is around $0.06$, and the variances differ by a few per cent). Two thousand values of $\zeta$ on $[0,2000]$ are strongly correlated, so this is a weak test; the book's Figure 6.2 uses $10\,000$ values over a longer range and $40\,000$ model samples, and there the statistic is about $0.01$.

### P6.3

In [ ]:
def zeta_grid(sigma, t, N=None):
    # Euler-Maclaurin: sum_{n<N} n^-s + N^{1-s}/(s-1) + N^-s/2 + s N^{-s-1}/12, error O(|s|^3 N^{-sigma-3})
    s = sigma + 1j * np.asarray(t)
    if N is None: N = int(max(50, np.max(np.abs(t)) / 2))
    out = np.zeros(s.shape, dtype=complex)
    for n in range(1, N):
        out += np.exp(-s * math.log(n))
    out += N ** (1 - s) / (s - 1) + 0.5 * N ** (-s) + s * N ** (-s - 1) / 12
    return out
for T in [1e3, 3e3]:
    t = np.linspace(T, 2 * T, int(10 * T) + 1)
    z = zeta_grid(0.7, t)
    print(f"T={T:.0f}: mean |zeta|^2 = {np.mean(np.abs(z)**2):.4f}   zeta(1.4) = {float(mpmath.zeta(1.4)):.4f}")

The agreement improves slowly: by the proof of Proposition 6.4 the error is a power $T^{-c}$ with small $c$ at $\sigma=0.7$, and $|\zeta|^2$ has a heavy tail.

### P6.4

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400, endpoint=False)
X = np.exp(1j * theta)
def phi(u, sigma=0.7):
    logabs = 0.0
    for p in P:
        lp = (-np.log(1 - X * p ** (-sigma))).real
        logabs += math.log(abs(np.mean(np.exp(1j * u * lp))) + 1e-300)
        if u * p ** (-sigma) < 0.01: break       # the remaining factors are 1 - O(u^2 p^-1.4)
    return logabs
for u in [5, 10, 20, 40, 60]:
    print(f"u={u:3d}: -log|phi(u)| = {-phi(u):8.2f},   u^(1/sigma)/log u = {u ** (1 / 0.7) / math.log(u):8.2f}")

The ratio of the two columns is not constant yet: it grows slowly with $u$, which is expected, since Theorem 6.8 only gives the order of magnitude and the $\log u$ factor and the finite range of $u$ blur the constant. What matters is visible: $\varphi$ decays like $\exp(-cu^{1/\sigma}/\log u)$, faster than any power, so the density is $C^\infty$. The cut-off in the loop is harmless: the remaining primes multiply $|\varphi|$ by $1-O(u^2\sum_{p>P}p^{-1.4})$. (Exact uniform sampling of $\theta$ on $400$ points integrates trigonometric polynomials of degree below $400$ exactly.)

### P6.5

In [ ]:
for t in [100.0, 1234.5, 3999.0]:
    print(t, zeta_grid(0.7, np.array([t]))[0], complex(mpmath.zeta(mpmath.mpc(0.7, t))))
t = np.arange(0, 2000, 0.01)
t0 = time.time(); z = np.concatenate([zeta_grid(0.7, t[i:i + 50000], N=1100) for i in range(0, len(t), 50000)])
print("vectorised grid of", len(t), "points:", round(time.time() - t0, 1), "s")
for target in [3, -1 + 2j, 0.1]:
    d = np.abs(z - target); i = d.argmin()
    print(f"target {target}: closest |zeta - z| = {d[i]:.4f} at t = {t[i]:.3f}")
frac = np.mean(np.abs(z[t > 100] - 3) < 0.2)
Lm = model_logzeta(0.7, 40000)
print("time within 0.2 of 3:", frac, "  model probability:", np.mean(np.abs(np.exp(Lm) - 3) < 0.2))

The evaluator agrees with mpmath to many digits, and evaluates $2\cdot10^5$ points in seconds, a thousand times faster per point. The targets $3$ and $0.1$ are approached within about $0.01$ before $t=2000$, but $-1+2i$ only within about $0.2$: Corollary 6.7 promises it will be approached as closely as we like, yet says nothing about how soon, and the model probability of that small region is tiny. The proportion of time near $3$ matches the model probability up to sampling error and the slow convergence in $T$.

### P6.6

In [ ]:
for sigma in [0.6, 0.55, 0.52, 0.51, 0.505]:
    v = 0.5 * sum(sum(p ** (-2 * k * sigma) / k**2 for k in range(1, 30)) for p in P)
    print(f"sigma={sigma}: v(sigma) (p<=1e5) = {v:.3f},  0.5*log(1/(sigma-1/2)) = {0.5 * math.log(1 / (sigma - 0.5)):.3f}")

Exercise 6.4 predicts that the difference between the two columns tends to a constant as $\sigma\to\frac12$, but here it does not settle (it goes from $-0.36$ to $-1.28$). The culprit is the truncation at $p\le10^5$: by the prime number theorem the missing primes contribute about $\frac12\int_{\log10^5}^\infty e^{-(2\sigma-1)v}\,\frac{dv}{v}\approx\frac12\log\frac{1}{(2\sigma-1)\log10^5}-\frac\gamma2$, which is no longer small once $2\sigma-1<1/\log10^5\approx0.087$, that is $\sigma<0.543$. The truncated sum stays bounded while the true $v(\sigma)$ blows up. A lesson worth keeping: a numerical check near a singularity needs a cut-off that moves with the singularity.